In [ ]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

act = new_client.activity

targets = {
    "COX2": "CHEMBL230",
    "5LOX": "CHEMBL215",
    "AChE": "CHEMBL220",
    "P-gp": "CHEMBL4302",
    "tubulin": "CHEMBL2095182",
    "gyrase": "CHEMBL3038482",
    "NorA": "CHEMBL5114",
}

ACTIVE_MAX = 10_000          # nM: exact IC50 <= 10 uM -> active
INACTIVE_EXACT_MIN = 50_000  # nM: exact IC50 >= 50 uM -> inactive
INACTIVE_CENSORED_MIN = 10_000  # nM: ">X" with X >= 10 uM -> inactive

FIELDS = ["molecule_chembl_id", "canonical_smiles", "standard_value", "standard_relation", "assay_chembl_id"]

def fetch(tid):
    recs = act.filter(
        target_chembl_id=tid,
        standard_type="IC50",
        standard_units="nM",
        standard_relation__in=["=", ">", ">="],
    ).only(FIELDS)
    df = pd.DataFrame(list(recs))
    if df.empty:
        return df
    df["standard_value"] = pd.to_numeric(df["standard_value"], errors="coerce")
    return df.dropna(subset=["standard_value", "canonical_smiles"])

def label(row):
    v, rel = row["standard_value"], row["standard_relation"]
    if rel == "=":
        if v <= ACTIVE_MAX:
            return 1
        if v >= INACTIVE_EXACT_MIN:
            return 0
    elif v >= INACTIVE_CENSORED_MIN:  # ">" or ">="
        return 0
    return None  # gray zone or ambiguous, dropped

def build(tid):
    df = fetch(tid)
    if df.empty:
        return df
    df["label"] = df.apply(label, axis=1)
    df = df.dropna(subset=["label"])
    # one row per compound; drop compounds with conflicting labels
    g = df.groupby("molecule_chembl_id").agg(
        smiles=("canonical_smiles", "first"),
        label=("label", "nunique"),
        y=("label", "first"),
        n_assays=("assay_chembl_id", "nunique"),
    )
    g = g[g["label"] == 1].drop(columns="label").reset_index()
    g["y"] = g["y"].astype(int)
    return g

datasets = {}
for name, tid in targets.items():
    d = build(tid)
    datasets[name] = d
    if d.empty:
        print(f"{name}: no data")
        continue
    n1, n0 = (d["y"] == 1).sum(), (d["y"] == 0).sum()
    print(f"{name}: {n1} actives, {n0} inactives, "f"active fraction {n1 / len(d):.2f}")